# F1 Top-10 Finish Prediction

This notebook is the assignment-facing source code view for the project. It uses the final model-ready dataset and demonstrates exploratory data analysis, preprocessing, temporal validation, and comparison of multiple predictive models.

## 1. Setup and Dataset Loading

The final dataset is a single CSV assembled from the raw public F1 sources. Each row is one driver in one race, and the target is `top10_finish`.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'scripts'))

DATA_PATH = PROJECT_ROOT / 'data' / 'final' / 'f1_top10_model_dataset.csv'
df = pd.read_csv(DATA_PATH)
df.shape

## 2. Exploratory Data Analysis

The following cells check dataset size, season coverage, target balance, missing values and available data types. These are the core EDA checks required before modelling.

In [ ]:
summary = {
    'rows': len(df),
    'columns': len(df.columns),
    'season_min': int(df['season'].min()),
    'season_max': int(df['season'].max()),
    'missing_values': int(df.isna().sum().sum()),
    'numeric_or_bool_columns': int(len(df.select_dtypes(include=['number', 'bool']).columns)),
    'categorical_columns': int(len(df.columns) - len(df.select_dtypes(include=['number', 'bool']).columns)),
}
summary

In [ ]:
df['top10_finish'].value_counts(normalize=True).rename('share').to_frame().join(
    df['top10_finish'].value_counts().rename('count')
)

In [ ]:
df.groupby('season').size().rename('driver_race_rows').to_frame().T

In [ ]:
eda_columns = ['grid', 'qualifying_position', 'driver_points_before_race', 'constructor_points_before_race', 'top10_finish']
df[[col for col in eda_columns if col in df.columns]].describe().T

## 3. Preprocessing and Temporal Split

The project uses a temporal split rather than a random split. The model trains on past seasons and validates on a future holdout season. Leakage columns such as final position, points and fastest lap information are removed from the training features.

In [ ]:
from train_model import TARGET, build_features, build_pipeline, latest_season_split

train_df, test_df = latest_season_split(df, test_season=2025, min_test_rows=200)
X_train = build_features(train_df)
y_train = train_df[TARGET].astype(int)
X_test = build_features(test_df)
y_test = test_df[TARGET].astype(int)

{'train_rows': len(train_df), 'test_rows': len(test_df), 'feature_count': X_train.shape[1]}

## 4. Train and Compare Predictive Models

The assignment asks for at least two predictive models. This notebook trains two representative models: logistic regression as an interpretable baseline and histogram gradient boosting as the current champion. The full project scripts compare additional models as well.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

def race_precision_at_10(test_rows, probabilities):
    scored = test_rows[['race_id', TARGET]].copy()
    scored['probability'] = probabilities
    scores = []
    for _, race in scored.groupby('race_id'):
        top10 = race.sort_values('probability', ascending=False).head(10)
        scores.append(float(top10[TARGET].mean()))
    return sum(scores) / len(scores)

rows = []
for model_name in ['logistic_regression', 'hist_gradient_boosting']:
    model = build_pipeline(X_train, model_name=model_name)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    rows.append({
        'model': model_name,
        'accuracy': accuracy_score(y_test, predictions),
        'precision': precision_score(y_test, predictions, zero_division=0),
        'recall': recall_score(y_test, predictions, zero_division=0),
        'f1': f1_score(y_test, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probabilities),
        'race_precision_at_10': race_precision_at_10(test_df, probabilities),
    })

pd.DataFrame(rows).sort_values('race_precision_at_10', ascending=False)

## 5. Full Project Results

The full script-based comparison trains five classifiers and stores the complete results under `outputs/`.

In [ ]:
comparison_path = PROJECT_ROOT / 'outputs' / 'model_comparison.csv'
pd.read_csv(comparison_path).sort_values('race_precision_at_10', ascending=False)

In [ ]:
metrics_path = PROJECT_ROOT / 'outputs' / 'metrics.json'
with metrics_path.open(encoding='utf-8') as file:
    json.load(file)

## 6. Optional Extension: Neural Network and 3D Visualization

The neural network is included as an extension, not as the main assignment result. The interactive 3D view can be opened from the project root with:

```powershell
python scripts/open_neural_network_3d.py
```